# 시뮬레이션 데이터 검증과 pose 출력: 시뮬레이션으로 학습한 side plate 모델

시뮬레이션(Isaac Sim)으로만 학습한 파이프라인을 **학습에 쓰지 않은 다른 시뮬레이션 데이터**(`side_plate_10k`, 5개 scene × 10장 = 50장)에 적용합니다.
이 데이터는 시뮬레이터의 정확한 정답과 비교하고, 마지막 §6에서 **Isaac Sim에서 side plate를 집기 위한 6D pose**를 출력합니다.

파이프라인은 다음 순서로 진행됩니다.

**Mask R-CNN detector → GDRN 방식 RGB-D pose 네트워크 → 깊이 맞춤 → ICP → 중복 제거 → 깊이 검증(오검출 제거) → GT 없는 confidence 판정**

**사용한 모델:** `runs/detector/best.pt`, `runs/pose/best.pt`. 둘 다 **시뮬레이션 데이터로만 학습**했습니다(실제 데이터 fine-tuning 모델은 쓰지 않음).

**데이터에 대한 주의:**
- 학습에 쓴 시뮬레이션 데이터(`data/bin_ds_side_plate_bp_sample`, 1236×1032)는 이제 디스크에 없습니다.
- 그래서 같은 부품과 bin을 **다른 카메라 설정**(976×730, fx 1182, 렌즈 왜곡 있음)으로 렌더링한 `~/FFB6D/data/side_plate_10k`를 씁니다.
- 학습 데이터와 장면이 겹치지 않으므로 처음 보는 데이터에 대한 평가입니다.

모든 숫자는 `runs/`의 추론 결과에서 매번 다시 계산합니다. 결론은 §7에 정리했습니다.

In [ ]:
import os, subprocess, sys, json
from pathlib import Path
if Path.cwd().name == "notebooks":                      # run from the project root
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

import numpy as np, pandas as pd, plotly.graph_objects as go, yaml
from IPython.display import display
from PIL import Image, ImageDraw, ImageFont
from scipy.ndimage import binary_dilation
from scipy.spatial.transform import Rotation

from bin_dataset import BinPoseDataset
from bin6d.config import Cfg, load_config
from bin6d.pose_data import ObjectModel
from bin6d.report import accuracy_table, detections_table, frame_index, load_frame, load_run, surface_check
from bin6d.viz import gt_overlay_sets, plotly_scene, render_2d, select

cfg = load_config("configs/default.yaml")
SRC_ROOT = "/home/joon/FFB6D/data/side_plate_10k"       # Isaac Sim renders, with lens distortion
SIM_ROOT = "data/side_plate_10k_undist"                  # the same frames resampled onto the pinhole K (undistort_bop.py)
SCENES = ["000000", "000001", "000002", "000003", "000004"]
STRIDE = 10                                              # every 10th view: views 0, 10, ..., 90 of each scene
DET_CKPT, POSE_CKPT = "runs/detector/best.pt", "runs/pose/best.pt"      # simulation-trained models only
DET, GTM = "detector masks (real pipeline)", "ground-truth masks"
RUNS = {DET: "runs/sim10k_full", GTM: "runs/sim10k_gtmask"}                         # current pipeline
RUNS_NOREJECT = {DET: "runs/sim10k_full_noreject"}                                    # same, without verification
RUNS_BEFORE = {DET: "runs/sim10k_full_noanchor", GTM: "runs/sim10k_gtmask_noanchor"}  # no anchor/dedup/verification

if not (Path(SIM_ROOT) / "shard_00" / "train_pbr" / SCENES[-1]).exists():
    subprocess.run([sys.executable, "undistort_bop.py", "--src", SRC_ROOT, "--dst", SIM_ROOT, "--scenes", *SCENES],
                   check=True)
sim = BinPoseDataset(SIM_ROOT, scenes=SCENES, bin_margin_mm=cfg.data.bin_margin_mm)
src = BinPoseDataset(SRC_ROOT, scenes=SCENES, bin_margin_mm=cfg.data.bin_margin_mm)
sidx = frame_index(sim)
model = ObjectModel(sim, cfg.data.obj_id, 0, 3000)
SHARD = sim.shards[0]
cam_json = json.load(open(SHARD / "camera.json"))
scene_cfg = yaml.safe_load(open(SHARD / "config_resolved.yaml"))
EVAL_VIEWS = [(s, v) for s in SCENES for v in range(0, 100, STRIDE)]
pd.set_option("display.width", 220)

def load_rejected(run_dir):
    """{(scene, view): [rejected detections]} written by infer.py with refine.reject_unverified."""
    out = {}
    for f in sorted((Path(run_dir) / "rejected").glob("*_*.json")):
        scene, view = f.stem.split("_")
        out[(scene, int(view))] = json.load(open(f))
    return out

def w2c(scene, view):
    """camera <- bin frame (scene_camera.json cam_R_w2c, cam_t_w2c; mm). The dataset's 'world' is the bin frame."""
    c = json.load(open(SHARD / "train_pbr" / scene / "scene_camera.json"))[str(view)]
    return np.asarray(c["cam_R_w2c"], float).reshape(3, 3), np.asarray(c["cam_t_w2c"], float)

S1, S2, S3 = "#2a78d6", "#eb6834", "#1baf7a"
def style(fig, title, height=360):
    fig.update_layout(template="plotly_white", title=title, height=height, font=dict(color="#52514e"),
                      margin=dict(l=60, r=20, t=60, b=40), legend=dict(orientation="h", y=1.12, x=0))
    fig.update_xaxes(gridcolor="#e1e0d9", linecolor="#c3c2b7")
    fig.update_yaxes(gridcolor="#e1e0d9", linecolor="#c3c2b7")
    return fig
N_PARTS = sum(len(sim.frame_meta(sidx[k])["objects_all"]["t"]) for k in EVAL_VIEWS)
print(f"sim: scenes {SCENES}, {len(EVAL_VIEWS)} evaluated frames, {N_PARTS} parts in the bin over them | "
      f"{cam_json['width']}x{cam_json['height']}, fx {cam_json['fx']}, lens distortion removed: {cam_json['undistorted_from']['dist']}")
print(f"models: detector {DET_CKPT}, pose {POSE_CKPT} (simulation only) | refine: depth_anchor={cfg.refine.depth_anchor}, "
      f"dedup_mm={cfg.refine.dedup_mm}, reject_unverified={cfg.refine.reject_unverified}")

## 1. 데이터

| | |
|---|---|
| 원본 | `~/FFB6D/data/side_plate_10k` (Isaac Sim, BOP 형식). scene 100개 × 100장 중 **scene 000000~000004의 view 0, 10, …, 90(50장)** 사용 |
| 카메라 | PhoXi 컬러 카메라 모델을 절반 해상도로 렌더링: **976×730, fx 1182**, depth 0.1mm 단위. **렌즈 왜곡 k1 = −0.3594가 렌더링에 들어 있음** |
| 왜곡 보정 | `undistort_bop.py`로 rgb, depth, 마스크를 핀홀 K로 다시 샘플링한 사본(`data/side_plate_10k_undist`)을 씀. 정답 pose와 K는 원래 핀홀 기준이라 그대로 |
| 정답 | 시뮬레이터의 정확한 pose(카메라 좌표). `poses/*.csv`에는 bin 좌표의 pose도 있음 |
| 장면 | 파란 bin에 부품 약 280~300개로 시작해, view가 진행되면서 비워 감(마지막 27~49개) |

**왜 왜곡 보정이 필요한가:** 파이프라인은 depth를 핀홀 K로만 3D로 바꿉니다. 왜곡된 원본에서는 가장자리 부품의 점군이 휘어서 CAD와 맞지 않습니다.
아래 표처럼 원본에서는 가장자리 부품의 정답 마스크가 CAD 실루엣 밖으로 벗어나고(0.94), depth가 CAD 뒤로 보이는 비율도 6.7%입니다. 보정 후에는 각각 1.00, 0.3%입니다.

In [ ]:
parts = pd.DataFrame([{"scene": s, "view": v, "parts in bin": len(o["t"]), "visible >= 500 px": int((o["visib_px"] >= 500).sum())}
                      for s, v in EVAL_VIEWS for o in [sim.frame_meta(sidx[(s, v)])["objects_all"]]])
print("parts in the bin per evaluated view (each scene is a picking run: the bin is emptied as the views go on)")
display(parts.pivot(index="scene", columns="view", values="parts in bin"))

# lens distortion: does the CAD projected with K at the GT pose cover each visible GT mask? (before / after resampling)
def gt_consistency(frames, scene, view):
    f = load_frame(frames, frame_index(frames)[(scene, view)])
    o, K, (h, w) = f["objects_all"], f["K"], f["depth"].shape
    rows = []
    for k in np.flatnonzero(o["visib_px"] >= 500):
        R, t = o["R"][k], o["t"][k]
        uv = (model.dense_points @ R.T + t) @ K.T
        u, v = np.round(uv[:, 0] / uv[:, 2]).astype(int), np.round(uv[:, 1] / uv[:, 2]).astype(int)
        ok = (u >= 0) & (u < w) & (v >= 0) & (v < h)
        sil = np.zeros((h, w), bool); sil[v[ok], u[ok]] = True
        sil = binary_dilation(sil, iterations=2)
        mk = f["inst"] == o["inst_id"][k]
        c = K @ t
        rows.append({"edge": np.hypot(*(c[:2] / c[2] - K[:2, 2])) > 300, "visible mask inside projected CAD": (mk & sil).sum() / mk.sum(),
                     "depth behind CAD (see-through)": surface_check(R, t, f["depth"], K, model)["violation"]})
    return pd.DataFrame(rows).groupby("edge").median().rename(index={False: "centre (< 300 px)", True: "edge (> 300 px)"})
print("GT consistency, scene 000000 view 0 (median over parts with >= 500 visible px)")
display(pd.concat({"original render": gt_consistency(src, "000000", 0), "undistorted": gt_consistency(sim, "000000", 0)}).round(3))

def strip(f, depth_range=(600, 850), h=300):
    lo, hi = depth_range
    dv = np.clip((f["depth"] - lo) / (hi - lo), 0, 1)
    dv = (np.where(f["valid"], 1 - dv, 0) * 255).astype(np.uint8)
    rng = np.random.default_rng(0); lut = rng.integers(40, 255, (f["inst"].max() + 1, 3)); lut[0] = 0
    tiles = [Image.fromarray(f["rgb"]), Image.fromarray(dv).convert("RGB"), Image.fromarray(lut[f["inst"]].astype(np.uint8))]
    w = int(h * f["rgb"].shape[1] / f["rgb"].shape[0])
    out = Image.new("RGB", (3 * w, h))
    for i, t in enumerate(tiles):
        out.paste(t.resize((w, h)), (w * i, 0))
    return out
print("scene 000000 view 0, original render (lens distortion): RGB | depth | GT visible masks")
display(strip(load_frame(src, frame_index(src)[("000000", 0)])))
for v in (0, 90):
    print(f"scene 000000 view {v}, undistorted ({len(sim.frame_meta(sidx[('000000', v)])['objects_all']['t'])} parts): RGB | depth | GT visible masks")
    display(strip(load_frame(sim, sidx[("000000", v)])))

### 학습 데이터와의 차이

In [ ]:
def frame_stats(frames, idx, key):
    f = load_frame(frames, idx[key])
    o, part = f["objects_all"], f["inst"] > 0
    vals = np.unique(f["depth"][f["valid"]])
    return {"camera -> part distance (mm)": float(np.median(o["t"][:, 2])),
            "visible part size (px, median)": float(np.median(o["visib_px"][o["visible"]])),
            "part diameter in px (at that distance)": float(model.diameter * f["K"][0, 0] / np.median(o["t"][:, 2])),
            "part pixels with depth": float((part & f["valid"]).sum() / part.sum()),
            "part brightness (0-255)": float(f["rgb"][part].mean()),
            "depth step (mm)": float(np.median(np.diff(vals))),
            "parts per frame": len(o["t"])}
cols = {f"this data ({len(EVAL_VIEWS)} frames)": pd.DataFrame([frame_stats(sim, sidx, k) for k in EVAL_VIEWS]).median(),
        # the training sim data (data/bin_ds_side_plate_bp_sample) is no longer on disk; values measured on its
        # validation scene 000009 in results_photoneo_summary.ipynb (1236x1032, fx ~1412)
        "training sim data (scene 000009, measured earlier)": pd.Series({
            "camera -> part distance (mm)": 715.7, "visible part size (px, median)": 3149.0,
            "part diameter in px (at that distance)": 55.7 * 1412.0 / 715.7, "part pixels with depth": 0.751,
            "part brightness (0-255)": 110.6, "depth step (mm)": 0.1, "parts per frame": 154.0})}
display(pd.DataFrame(cols).round(3))

- **카메라가 다릅니다.** 부품이 이미지에서 약 0.8배 작게 보입니다(지름 약 89px, 학습 약 110px).
  detector는 원본 해상도로 학습했지만, 시험 결과 입력 배율 1.0(기본값)이 1.12보다 나았습니다.
- **부품이 더 어둡게** 렌더링되고(밝기 78, 학습 111), depth가 있는 부품 픽셀도 조금 적습니다(70%, 학습 75%).
- 장당 부품 수(약 160개)와 카메라 거리(약 740mm)는 비슷합니다.

## 2. 추론 실행

`infer.py`를 실행해 결과(JSON)를 `runs/`에 저장합니다. 결과가 이미 있으면 다시 실행하지 않습니다. 모델이나 설정을 바꿨다면 `RERUN = True`로 바꾸세요.
깊이 검증으로 걸러진 검출은 각 결과 폴더의 `rejected/`에 저장됩니다. 5가지 모두 시뮬레이션 학습 모델(`runs/detector/best.pt`, `runs/pose/best.pt`)을 씁니다.

| 결과 폴더 | 마스크 | 깊이 맞춤, 중복 제거 | 깊이 검증 | 시간 (GB10) |
|---|---|---|---|---|
| `runs/sim10k_full`, `runs/sim10k_gtmask` | detector / GT | 켬 | 켬 (현재 설정) | 약 5분씩 |
| `runs/sim10k_full_noreject` | detector | 켬 | 끔 (비교용) | 약 5분 |
| `runs/sim10k_*_noanchor` | detector / GT | 끔 | 끔 (비교용) | 약 5분씩 |

```bash
.venv/bin/python undistort_bop.py --src ~/FFB6D/data/side_plate_10k --dst data/side_plate_10k_undist --scenes 000000 000001 000002 000003 000004
.venv/bin/python infer.py --set data.root=data/side_plate_10k_undist --scenes 000000 000001 000002 000003 000004 --view-stride 10 \
    --detector runs/detector/best.pt --pose runs/pose/best.pt --no-viz --out runs/sim10k_full
```

In [ ]:
RERUN = False
BASE = ["--set", f"data.root={SIM_ROOT}", "--scenes", *SCENES, "--view-stride", str(STRIDE), "--no-viz",
        "--detector", DET_CKPT, "--pose", POSE_CKPT]
NOREJECT = ["--set", "refine.reject_unverified=false"]
OFF = ["--set", "refine.depth_anchor=false", "--set", "refine.dedup_mm=0"] + NOREJECT
jobs = [(RUNS[DET], BASE), (RUNS[GTM], BASE + ["--gt-masks"]), (RUNS_NOREJECT[DET], BASE + NOREJECT),
        (RUNS_BEFORE[DET], BASE + OFF), (RUNS_BEFORE[GTM], BASE + OFF + ["--gt-masks"])]
for out, args in jobs:
    if RERUN or not list(Path(out).glob("*_*.json")):
        cmd = [sys.executable, "infer.py", "--out", out] + args
        print(" ".join(cmd))
        subprocess.run(cmd, check=True, env={**os.environ, "HF_HUB_OFFLINE": "1"})
    print(f"{out}: {len(list(Path(out).glob('*_*.json')))} frames")

## 3. 정답 대비 정확도

**ADD-S**는 예측 pose와 정답 pose에서 CAD 표면 점들 사이의 평균 거리입니다(대칭 중 최솟값). **centre err**는 위치 오차, **tilt**는 판 법선 사이의 각도입니다.
`n`은 정답과 매칭된 검출 수입니다(마스크 IoU ≥ 0.5). 5.6mm는 부품 지름의 10%로, BOP 표준 기준입니다.
**+ depth anchor** 줄은 네트워크 pose를 실측 depth에 맞춘 뒤, ICP 전의 결과입니다.
시뮬레이터 정답이라 실제 데이터(의사 정답)와 달리 0.1mm 아래의 차이도 잴 수 있습니다.

In [ ]:
tables = {name: detections_table(load_run(path, cfg.refine), sim, model) for name, path in RUNS.items()}
for name, df in tables.items():
    print(f"\n{name}: {len(df)} outputs, {df.matched.sum()} matched to GT ({N_PARTS} parts in the bin, many fully hidden)")
    display(accuracy_table(df, diameter=model.diameter))

**읽는 법:** 네트워크만으로는 1mm 이내가 6%뿐이지만, 깊이 맞춤과 ICP를 거치면 **98%가 1mm 이내**이고 위치 오차 중앙값은 0.04mm입니다.
네트워크의 역할은 ICP가 수렴할 범위 안에 초기값을 넣는 것입니다.

### 단계별 비교

열은 왼쪽부터 기능을 하나씩 켠 결과입니다.
- **before**: 깊이 맞춤, 중복 제거, 깊이 검증을 모두 끔
- **anchor + dedup**: 깊이 맞춤과 중복 제거만 켬
- **+ verification**: 현재 설정

**confident인데 틀림**은 confident로 표시됐지만 정답과 1mm 이상 다른 pose입니다. 로봇이 잘못 집을 수 있는 위험한 경우입니다.

In [ ]:
def summary(df, rejected=None):
    m = df[df.matched]
    c = m[m.confident]
    n_frames = df.groupby(["scene", "view"]).ngroups
    out = {"outputs per frame": round(len(df) / n_frames, 1),
           "matched": len(m),
           "false detections in output": int((~df.matched).sum()),
           "ICP accepted (share of matched)": f"{m.icp_refined.mean():.0%}",
           "confident per frame": round(df.confident.sum() / n_frames, 1),
           "confident (share of matched)": f"{m.confident.mean():.0%}",
           "confident: ADD-S < 1 mm": f"{(c.final_add_s < 1).mean():.1%}",
           "confident but wrong (>= 1 mm)": int((c.final_add_s >= 1).sum()),
           "confident false detections": int((df.confident & ~df.matched).sum()),
           "confident: centre err median (mm)": round(c.final_centre_mm.median(), 3),
           "confident: tilt median (deg)": round(c.final_tilt_deg.median(), 3)}
    if rejected is not None:
        rj = [d for ds in rejected.values() for d in ds]
        out["rejected by verification (of them real parts)"] = f"{len(rj)} ({sum(d['gt_inst_id'] is not None for d in rj)})"
    return out

compare = {}
for name in RUNS:
    cols = {"before": summary(detections_table(load_run(RUNS_BEFORE[name], cfg.refine), sim, model))}
    if name in RUNS_NOREJECT:
        cols["anchor + dedup"] = summary(detections_table(load_run(RUNS_NOREJECT[name], cfg.refine), sim, model))
    cols["+ verification (current)"] = summary(tables[name], load_rejected(RUNS[name]))
    compare[name] = pd.DataFrame(cols).fillna("")
    print(name); display(compare[name])

**읽는 법:**
- **깊이 맞춤:** confident가 장당 56개에서 71개로 늘었습니다. 대신 confident인데 틀린 pose도 22개에서 67개로 늘었습니다(1mm 이내 99.2% → 98.1%).
  학습 데이터에서 본 것과 같은 경향입니다. 원래 깊이가 틀려서 ICP가 거부하던 pose 중 **회전도 틀린 것**까지 ICP 범위에 들어오고, 평평한 면이 depth에 잘 맞아 통과하기 때문입니다.
- **깊이 검증:** 오검출이 202개에서 31개로 줄었습니다. confident pose 수와 정확도는 그대로입니다.

### 오검출 제거: 깊이 검증

In [ ]:
def fp_kind(dets, objs, matched_ids):
    """Detections not matched to GT by mask IoU >= 0.5. A pose within 2 mm of a part that no other output
    covers is that part with a partial mask (correct pose, IoU < 0.5): not a false detection for picking."""
    rows = []
    for d in dets:
        if d["gt_inst_id"] is None:
            dist = np.linalg.norm(objs["t"] - np.asarray(d["t"]), axis=1)
            k = int(dist.argmin())
            if dist[k] < 2 and int(objs["inst_id"][k]) not in matched_ids:
                kind = "real part, partial mask (pose correct)"
            elif np.linalg.norm(objs["t"][k, :2] - np.asarray(d["t"])[:2]) < 40:
                kind = "false: duplicate / partial on a part"
            else:
                kind = "false: no part nearby (rim / outside bin)"
            rows.append({"kind": kind})
    return rows

rows = []
for label, path in (("anchor + dedup (no verification)", RUNS_NOREJECT[DET]), ("+ verification: output", RUNS[DET])):
    for (s, v), dets in load_run(path).items():
        ids = {d["gt_inst_id"] for d in dets if d["gt_inst_id"] is not None}
        rows += [{"run": label, **r} for r in fp_kind(dets, sim.frame_meta(sidx[(s, v)])["objects_all"], ids)]
print("outputs not matched to GT by mask IoU (detector masks)")
display(pd.DataFrame(rows).groupby(["run", "kind"]).size().rename("count").to_frame() if rows else "none")

rej = pd.DataFrame([{"reason": d["reject"], "real part": d["gt_inst_id"] is not None, "ADD-S if real (mm)": d.get("add_s_refined"),
                     "depth points": d["n_depth_points"]}
                    for ds in load_rejected(RUNS[DET]).values() for d in ds])
print("rejected by verification (detector masks)")
display(rej.groupby(["reason", "real part"]).size().unstack(fill_value=0)
        .rename(columns={False: "false detection", True: "real part"}))
lost = rej[rej["real part"]]
print(f"real parts rejected: {len(lost)}; their pose error (ADD-S): median {lost['ADD-S if real (mm)'].median():.1f} mm, "
      f"< 1 mm: {(lost['ADD-S if real (mm)'] < 1).mean():.0%}; depth points median {lost['depth points'].median():.0f}")

# coverage of the parts a robot could see: visible >= 500 px
nore, ver = load_run(RUNS_NOREJECT[DET], cfg.refine), load_run(RUNS[DET], cfg.refine)
rows = []
for s, v in EVAL_VIEWS:
    o = sim.frame_meta(sidx[(s, v)])["objects_all"]
    found = {d["gt_inst_id"] for d in nore[(s, v)] if d["gt_inst_id"] is not None}
    out = {d["gt_inst_id"]: d for d in ver[(s, v)] if d["gt_inst_id"] is not None}
    for k, iid in enumerate(o["inst_id"]):
        if o["visib_px"][k] < 500:
            continue
        d = out.get(int(iid))
        rows.append({"visible px": o["visib_px"][k], "found by detector": int(iid) in found,
                     "output (verified)": d is not None, "confident < 1 mm": bool(d and d["confident"] and d["add_s_refined"] < 1)})
vis = pd.DataFrame(rows)
vis["visible"] = pd.cut(vis["visible px"], [499, 2000, 5000, 1e9], labels=["500-2000 px", "2000-5000 px", "> 5000 px (mostly free)"])
g = vis.groupby("visible", observed=True)
print("parts with >= 500 visible px")
display(pd.concat([g.size().rename("parts"), g[["found by detector", "output (verified)", "confident < 1 mm"]].mean().map("{:.1%}".format)], axis=1))

**결과:**
- **출력에 남은 "정답과 매칭 안 됨" 31개 중 30개는 실제 부품입니다.** pose가 정답과 2mm 이내인데, 마스크가 부품 일부만 덮어 IoU가 0.5 미만이었습니다. 진짜 오검출은 1개(중복 마스크)입니다.
- **검증에서 빠진 실제 부품 1329개는 모두 pose가 틀린 것입니다.** ADD-S 중앙값 9.4mm이고, 1mm 이내는 0%입니다. 쌓인 bin에서 많이 가려진 부품이 대부분입니다.
- **가려지지 않은 부품(보이는 픽셀 5000 이상)은 96%가 confident이고 1mm 이내**입니다. 반쯤 가려진 부품(2000~5000px)은 79%, 많이 가려진 부품(500~2000px)은 14%입니다.
  로봇은 위쪽의 잘 보이는 부품부터 집으므로, 집을 대상은 충분히 남습니다.

## 4. 깊이 맞춤의 효과: 네트워크의 깊이(z) 오차

네트워크에 들어가는 depth는 **부품 표면 중앙값을 뺀 상대값**이라, 네트워크는 부품이 몇 mm 떨어져 있는지 겉보기 크기로만 추정합니다.
ICP는 보정량이 15mm를 넘으면 결과를 거부합니다.

In [ ]:
def depth_errors(run, frames):
    idx = frame_index(frames)
    rows = []
    for (s, v), dets in run.items():
        o = frames.frame_meta(idx[(s, v)])["objects_all"]
        g = {int(i): k for k, i in enumerate(o["inst_id"])}
        for d in dets:
            if d["gt_inst_id"] is None:
                continue
            tg = o["t"][g[d["gt_inst_id"]]]
            row = {"network dz (mm)": d["t_rgbd"][2] - tg[2], "ICP accepted": d["icp_refined"]}
            if "t_anchored" in d:
                row["anchored dz (mm)"] = d["t_anchored"][2] - tg[2]
            rows.append(row)
    return pd.DataFrame(rows)

def dz_summary(e, col):
    s = e[col]
    return {"median dz (mm, + = too far)": round(s.median(), 1), "too far": f"{(s > 0).mean():.0%}",
            "|dz| median (mm)": round(s.abs().median(), 1), "|dz| > 15 mm (ICP limit)": f"{(s.abs() > 15).mean():.0%}"}

e_now = depth_errors(load_run(RUNS[GTM], cfg.refine), sim)
e_before = depth_errors(load_run(RUNS_BEFORE[GTM], cfg.refine), sim)
print("depth error vs GT, GT masks")
display(pd.DataFrame({"network": dz_summary(e_now, "network dz (mm)"), "after anchor": dz_summary(e_now, "anchored dz (mm)")}))
g = e_before.groupby("ICP accepted")["network dz (mm)"]
print("before anchoring: network |dz| by ICP outcome")
display(pd.DataFrame({"detections": g.size(), "|dz| median (mm)": g.apply(lambda s: s.abs().median()).round(1),
                      "|dz| > 15 mm": g.apply(lambda s: f"{(s.abs() > 15).mean():.0%}")}))

fig = go.Figure()
for col, c, label in (("network dz (mm)", S2, "network"), ("anchored dz (mm)", S3, "after depth anchor")):
    fig.add_trace(go.Histogram(x=e_now[col], name=label, marker_color=c, xbins=dict(start=-40, end=40, size=1), opacity=0.8))
fig.add_vline(x=15, line_dash="dash", line_color="#52514e", annotation_text="ICP limit 15 mm")
fig.add_vline(x=-15, line_dash="dash", line_color="#52514e")
style(fig, "Depth error before ICP (GT masks)").update_layout(barmode="overlay")
fig.update_xaxes(title="dz = predicted - GT (mm), + = too far").update_yaxes(title="detections")
fig.show()

- **이 데이터에서도 깊이 오차는 무작위입니다**(멀게 46%, 가깝게 54%). 카메라가 학습과 달라도 실제 Photoneo(95%가 멀게)처럼 한쪽으로 치우치지 않습니다.
  다만 7%는 15mm를 넘습니다.
- 깊이 맞춤 전에 ICP가 거부된 검출은 깊이 오차 중앙값이 14mm였습니다. 깊이 맞춤 후에는 중앙값 0.4mm이고, 15mm를 넘는 것은 없습니다.

### confidence 기준 비교

In [ ]:
def variant(path, min_fit, ratio, cap):
    rc = Cfg(dict(cfg.refine)); rc.confidence_min_fitness, rc.confidence_rmse_ratio, rc.confidence_max_rmse_mm = min_fit, ratio, cap
    d = detections_table(load_run(path, rc), sim, model)
    c = d[d.confident & d.matched]
    return {"confident": int(d.confident.sum()), "per frame": round(d.confident.sum() / d.groupby(["scene", "view"]).ngroups, 1),
            "confident false detections": int((d.confident & ~d.matched).sum()),
            "ADD-S < 1 mm": f"{(c.final_add_s < 1).mean():.1%}" if len(c) else "-", "wrong (>= 1 mm)": int((c.final_add_s >= 1).sum())}
for name, path in RUNS.items():
    print(name)
    display(pd.DataFrame({"fixed 0.8 / 1.0 mm": variant(path, 0.8, 1e9, 1.0), "fixed 0.9 / 0.5 mm": variant(path, 0.9, 1e9, 0.5),
                          "relative 0.9 / 1.3x median (cap 1.5 mm)": variant(path, 0.9, 1.3, 1.5)}).T)
acc = tables[DET][tables[DET].icp_refined]
print(f"ICP-accepted fits: fitness median {acc.icp_fitness.median():.3f}, fit error (RMSE) median {acc.icp_rmse.median():.3f} mm")

fit 오차 중앙값은 0.44mm입니다. 고정 기준 0.5mm는 이 값 바로 위라 장당 약 7개의 pose를 잃습니다. 프레임마다 기준을 맞추는 상대 기준(현재 설정)을 유지합니다.

### scene별 결과와 bin을 비워 가는 동안

In [ ]:
for name in RUNS:
    df = tables[name]
    pf = df.groupby(["scene", "view"]).agg(outputs=("matched", "size"), confident=("confident", "sum"))
    c = df[df.matched & df.confident]
    pf["confident correct (< 1 mm)"] = c.groupby(["scene", "view"]).final_add_s.apply(lambda s: int((s < 1).sum()))
    pf["confident wrong"] = c.groupby(["scene", "view"]).final_add_s.apply(lambda s: int((s >= 1).sum()))
    pf["parts in bin"] = [len(sim.frame_meta(sidx[k])["objects_all"]["t"]) for k in pf.index]
    pf = pf.fillna(0).astype(int)
    print(name)
    display(pf.groupby("scene").sum().assign(**{"confident correct": lambda d: (d["confident correct (< 1 mm)"] / d.confident).map("{:.1%}".format)}))
    if name == DET:
        fig = go.Figure()
        for s, col in zip(SCENES, [S1, S2, S3, "#8a63d2", "#d6a12a"]):
            d = pf.loc[s]
            fig.add_trace(go.Scatter(x=d.index, y=d.confident, name=f"scene {s}", mode="lines+markers", line=dict(color=col)))
            fig.add_trace(go.Scatter(x=d.index, y=d["parts in bin"], mode="lines", line=dict(color=col, dash="dot"), showlegend=False))
        style(fig, "Confident poses (solid) and parts in the bin (dotted) as each scene is emptied, detector masks")
        fig.update_xaxes(title="view").update_yaxes(title="count", rangemode="tozero")
        fig.show()

c = tables[DET][tables[DET].matched & tables[DET].confident]
wrong = c[c.final_add_s >= 1]
print(f"confident but wrong (ADD-S >= 1 mm), detector masks: {len(wrong)} of {len(c)}; what is wrong in them:")
display(wrong[["final_add_s", "final_centre_mm", "final_tilt_deg", "icp_fitness", "icp_rmse"]]
        .describe(percentiles=[0.5, 0.9]).loc[["count", "50%", "90%", "max"]].round(2))

- **bin을 비워 가는 동안 confident pose가 계속 나옵니다.** 부품 수가 줄면 confident 수도 함께 줄지만, 마지막(27~49개)까지 남아 있습니다.
- **confident인데 틀린 67개는 대부분 "평면 안 회전" 오류입니다.** 위치 오차 중앙값 0.5mm(90%가 1.5mm 이하), 기울기 중앙값 0.6°(90%가 3.2° 이하)로, 위치와 판 방향은 거의 맞습니다.
  판이 평평해서 depth로는 회전을 구분하기 어렵기 때문입니다. 부품을 중심에서 집는다면 영향이 작습니다.

### 부품 자세별 결과

In [ ]:
# every part with >= 500 visible px: found by the detector (before verification) / output after verification / confident
rows = []
for s, v in EVAL_VIEWS:
    o = sim.frame_meta(sidx[(s, v)])["objects_all"]
    found = {d["gt_inst_id"] for d in nore[(s, v)] if d["gt_inst_id"] is not None}
    out = {d["gt_inst_id"]: d for d in ver[(s, v)] if d["gt_inst_id"] is not None}
    for k, iid in enumerate(o["inst_id"]):
        if o["visib_px"][k] < 500:
            continue
        d = out.get(int(iid))
        rows.append({"pose": "CAD z away from camera" if o["R"][k][2, 2] > 0 else "CAD z towards camera",
                     "found by detector": int(iid) in found, "output (verified)": d is not None,
                     "confident": bool(d and d["confident"]),
                     "confident < 1 mm": bool(d and d["confident"] and d["add_s_refined"] < 1)})
pp = pd.DataFrame(rows)
g = pp.groupby("pose")
display(pd.concat([g.size().rename("parts"), g[["found by detector", "output (verified)", "confident", "confident < 1 mm"]]
                   .mean().map("{:.1%}".format)], axis=1))

시뮬레이션에서는 자세에 따른 차이가 거의 없습니다(confident 63.6% vs 60.4%). 실제 Photoneo에서 한쪽 자세를 놓치던 문제는 실제 겉모습 때문이었고, 모델 자체의 약점은 아닙니다.

## 5. 시각 확인

2D 오버레이: 각 검출의 3D 박스를 투영해 그렸습니다.
- **초록** = confident이고 정답과 1mm 이내
- **빨강** = confident인데 틀림
- **주황** = confident 아님
- **얇은 흰색** = pose가 맞지 않은 곳의 정답
- **자홍** = detector가 놓친 부품

confident인 부품에는 xyz 축을 그립니다(x 빨강, y 초록, z 파랑).
"problems"는 문제가 있는 것만, "confident"는 confident인 것만 보여 줍니다.

In [ ]:
def overlay(scene, view, path, label, show="all", width=1000):
    f = load_frame(sim, sidx[(scene, view)])
    dets = load_run(path, cfg.refine)[(scene, view)]
    gt = gt_overlay_sets(f["objects_all"], dets, cfg.pose.min_visib_px)
    dets, gt = select(dets, gt, show)
    img = render_2d(f["rgb"], f["K"], dets, model, gt, labels="wrong", title=f"scene {scene} view {view} ({label}): {show}")
    display(img.resize((width, int(width * img.height / img.width))))
overlay("000000", 0, RUNS_NOREJECT[DET], "detector, without verification", "problems")
overlay("000000", 0, RUNS[DET], "detector, + verification", "confident")
overlay("000000", 0, RUNS[DET], "detector, + verification", "problems")
overlay("000000", 90, RUNS[DET], "detector, + verification", "all")

**대화형 3D** (드래그로 회전, 스크롤로 확대, 박스에 마우스를 올리면 fitness, fit 오차, ADD-S 표시).

In [ ]:
s, v = "000000", 50
f = load_frame(sim, sidx[(s, v)])
dets = load_run(RUNS[DET], cfg.refine)[(s, v)]
gt = gt_overlay_sets(f["objects_all"], dets, cfg.pose.min_visib_px)
plotly_scene(f["rgb"], f["xyz"], f["valid"], dets, model, gt, max_points=40000,
             title=f"scene {s} view {v}: detections over the point cloud, camera frame (mm)").show()

## 6. Isaac Sim에서 집을 부품의 6D pose 출력

한 프레임에서 **집을 후보 부품을 골라 6D pose를 출력**합니다. 선택 기준은 정답 없이 계산할 수 있는 것만 씁니다.
1. **confident**(ICP fit과 깊이 검증 통과)
2. **위가 비어 있음**: 부품의 보이는 CAD 면 중 다른 물체에 덮인 비율이 `MAX_COVERED`(5%) 이하. 부품 자신의 허브가 가리는 면은 빼고 계산합니다(`covered_share`).
3. 그중 **높은 순서**(bin 좌표 z)로 `TOP_N`(8)개

**출력 위치:** 부품 중심(CAD bbox 중심)이 아니라, **부품 축 위에서 위쪽을 향한 끝면의 중심**(그리퍼가 닿는 곳)입니다.
- 부품이 허브를 위로 하고 있으면 허브 끝(CAD +z 끝)이고, 뒤집혀 있으면 평평한 판 면(CAD −z 끝)입니다. 어느 쪽인지는 `end` 열에 표시합니다.
- 부품 중심 좌표는 CSV의 `*_centre_x/y/z_mm` 열에 따로 남깁니다.
- 그림의 노란 원이 이 출력 위치입니다.

**좌표계:**
- **Isaac Sim 월드:** 위치는 m, 자세는 **roll, pitch, yaw(도)**입니다. 고정축 x → y → z 순서(`R = Rz(yaw)·Ry(pitch)·Rx(roll)`)이고, 데이터의 `poses/*.csv` `*_roll/pitch/yaw_deg` 열과 같은 정의입니다.
  `config_resolved.yaml`의 bin 위치(`scene.bin.pos_m = (1.0, 0, 0)`, 회전 0)를 bin 좌표에 더해서 구합니다.
- **bin 좌표:** `scene_camera.json`의 월드(`cam_R_w2c`, `cam_t_w2c`)이고, 데이터의 `poses/*.csv` `bin_*` 열과 같습니다. bin 바닥 중심이 원점이고 z가 위쪽입니다(mm).
  변환식은 `p_bin = R_w2c^T (p_cam − t_w2c)`, `R_bin = R_w2c^T R_cam`이고, 데이터의 정답 CSV와 소수점까지 일치하는 것을 확인했습니다.
- **카메라 좌표:** 네트워크와 ICP가 내는 그대로입니다(mm).
- **부품 좌표:** CAD 원점은 bbox 중심이고 z축이 판의 축입니다. 부품은 z축 둘레 **180° 대칭**이라, z축으로 180° 돌린 pose도 같은 pose입니다.

그림은 선택된 부품의 CAD 표면 점을 실측 depth와 비교한 색으로 보여 줍니다.
- 초록 = depth와 일치
- 빨강 = depth가 CAD 뒤에 있음(그 자리에 부품이 없음)
- 파랑 = 앞에 무언가 있음(허브 등 자기 가림 포함)

결과는 `runs/sim10k_full/pick_poses/<scene>_<view>.csv`(선택된 것)와 `..._all_confident.csv`(confident 전체)로 저장됩니다.
`SCENE`, `VIEW`, `TOP_N`, `MAX_COVERED`를 바꿔 다른 프레임에서도 출력할 수 있습니다.

In [ ]:
SCENE, VIEW = "000000", 0          # frame to pick from
TOP_N = 8                          # how many poses to show / export
MAX_COVERED = 0.05                 # share of the part's visible CAD surface with something lying on top of it
TOL_MM = 3.0                       # CAD-vs-depth agreement tolerance for the overlay and the checks

bin_pos_mm = 1000.0 * np.asarray(scene_cfg["scene"]["bin"]["pos_m"], float)       # bin frame -> Isaac Sim world
bin_rot = Rotation.from_euler("xyz", scene_cfg["scene"]["bin"]["rpy_deg"], degrees=True)
R_w2c, t_w2c = w2c(SCENE, VIEW)
f = load_frame(sim, sidx[(SCENE, VIEW)])
dets = load_run(RUNS[DET], cfg.refine)[(SCENE, VIEW)]
gt_tab = tables[DET].set_index(["scene", "view", "det"])

from bin6d.geometry import sample_surface
RENDER_PTS = sample_surface(model.verts, model.faces, 60000, seed=5, return_normals=True)

def covered_share(R, t, depth, K, tol=TOL_MM):
    """Share of the part's camera-visible CAD surface with measured depth in front of it by > tol (something lies on
    the part). Points hidden by the part's own geometry (e.g. the plate behind its hub) are excluded first, so a free
    part reads ~0 whichever side faces up."""
    h, w = depth.shape
    R, t = np.asarray(R, float), np.asarray(t, float)
    pts, nrm = RENDER_PTS[0] @ R.T + t, RENDER_PTS[1] @ R.T
    pts = pts[(nrm * pts).sum(1) < 0]
    uv = pts @ K.T
    u, v = np.round(uv[:, 0] / uv[:, 2]).astype(int), np.round(uv[:, 1] / uv[:, 2]).astype(int)
    ok = (u >= 0) & (u < w) & (v >= 0) & (v < h)
    u, v, z = u[ok], v[ok], pts[ok, 2]
    zbuf = np.full(h * w, np.inf)
    np.minimum.at(zbuf, v * w + u, z)
    seen = (z <= zbuf[v * w + u] + 1.0) & (depth[v, u] > 0)       # visible w.r.t. its own geometry, has depth
    return float((depth[v, u][seen] - z[seen] < -tol).mean()) if seen.any() else 1.0

# the two ends of the part along its axis (CAD z): +z = hub end (ring r 6.9-9.1 mm), -z = flat plate face
Z_ENDS = {"hub": float(model.xyz_min[2] + model.xyz_size[2]), "flat face": float(model.xyz_min[2])}

def top_end(R_b, t_b):
    """Point on the part axis at the end face that points up in the bin frame (where a gripper touches)."""
    ends = {name: R_b @ np.array([0.0, 0.0, z]) + t_b for name, z in Z_ENDS.items()}
    name = max(ends, key=lambda k: ends[k][2])
    return name, ends[name]

def rpy_deg(R):
    """roll, pitch, yaw in degrees, fixed axes x -> y -> z: R = Rz(yaw) Ry(pitch) Rx(roll)
    (the definition of the *_roll/pitch/yaw_deg columns in the dataset's poses/*.csv)."""
    return Rotation.from_matrix(R).as_euler("xyz", degrees=True)

rows = []
for n, d in enumerate(dets):
    if not d["confident"]:
        continue
    R_c, t_c = np.asarray(d["R"]), np.asarray(d["t"])
    R_b, t_b = R_w2c.T @ R_c, R_w2c.T @ (t_c - t_w2c)                      # bin frame (= dataset world)
    R_i, t_i = bin_rot.as_matrix() @ R_b, bin_rot.apply(t_b) + bin_pos_mm  # Isaac Sim world
    end_name, e_b = top_end(R_b, t_b)                                       # output position: the top end face
    ends = {"cam": R_w2c @ e_b + t_w2c, "bin": e_b, "isaac": bin_rot.apply(e_b) + bin_pos_mm}
    centres = {"cam": t_c, "bin": t_b, "isaac": t_i}
    sc = surface_check(R_c, t_c, f["depth"], f["K"], model, TOL_MM)
    row = {"det": n, "end": end_name, "height_mm": e_b[2], "axis_tilt_deg": np.degrees(np.arccos(abs(R_b[2, 2]))),
           "covered": covered_share(R_c, t_c, f["depth"], f["K"]), "depth_agree": sc["support"], "fitness": d["icp_fitness"], "rmse_mm": d["icp_rmse"]}
    for tag, R in (("cam", R_c), ("bin", R_b), ("isaac", R_i)):
        roll, pitch, yaw = rpy_deg(R)
        e, c = ends[tag], centres[tag]
        row.update({f"{tag}_x_mm": e[0], f"{tag}_y_mm": e[1], f"{tag}_z_mm": e[2],
                    f"{tag}_roll_deg": roll, f"{tag}_pitch_deg": pitch, f"{tag}_yaw_deg": yaw,
                    f"{tag}_centre_x_mm": c[0], f"{tag}_centre_y_mm": c[1], f"{tag}_centre_z_mm": c[2]})
    g = gt_tab.loc[(SCENE, VIEW, n)]
    if d["gt_inst_id"] is not None:                                         # same end point from the GT pose
        o = f["objects_all"]
        k = int(np.flatnonzero(o["inst_id"] == d["gt_inst_id"])[0])
        _, e_gt = top_end(R_w2c.T @ o["R"][k], R_w2c.T @ (o["t"][k] - t_w2c))
        row["gt_end_err_mm"] = float(np.linalg.norm(e_b - e_gt))
    row.update({"gt_centre_err_mm": g.final_centre_mm if g.matched else np.nan,
                "gt_tilt_err_deg": g.final_tilt_deg if g.matched else np.nan,
                "gt_add_s_mm": g.final_add_s if g.matched else np.nan})
    rows.append(row)
cand = pd.DataFrame(rows)
picks = (cand[cand.covered <= MAX_COVERED].sort_values("height_mm", ascending=False).head(TOP_N)
         .reset_index(drop=True))
picks.index = picks.index + 1
picks.index.name = "pick"
print(f"scene {SCENE} view {VIEW}: {len(dets)} verified poses, {len(cand)} confident, "
      f"{int((cand.covered <= MAX_COVERED).sum())} with <= {MAX_COVERED:.0%} of the visible surface covered -> top {len(picks)} by height")

In [ ]:
def draw_picks(f, dets, picks, width=1100, pad=60):
    """Selected poses like a picking UI: CAD surface points coloured by agreement with the measured depth
    (green = on the depth within TOL_MM, red = depth behind the CAD (see-through), blue = something in front),
    xyz axes (x red, y green, z blue) and the pick number."""
    K, depth, h, w = f["K"], f["depth"], *f["depth"].shape
    base = Image.fromarray((np.asarray(Image.fromarray(f["rgb"]).convert("L"), float) * 0.55).astype(np.uint8)).convert("RGB")
    px = np.asarray(base).copy()
    boxes = []
    for _, p in picks.iterrows():
        d = dets[int(p.det)]
        R, t = np.asarray(d["R"]), np.asarray(d["t"])
        pts = model.dense_points @ R.T + t
        pts = pts[((model.dense_normals @ R.T) * pts).sum(1) < 0]
        uv = pts @ K.T
        u, v = np.round(uv[:, 0] / uv[:, 2]).astype(int), np.round(uv[:, 1] / uv[:, 2]).astype(int)
        ok = (u >= 0) & (u < w) & (v >= 0) & (v < h)
        u, v, z = u[ok], v[ok], pts[ok, 2]
        dm = depth[v, u]
        col = np.where(((dm > 0) & (dm - z > TOL_MM))[:, None], [235, 60, 60],
                       np.where(((dm > 0) & (dm - z < -TOL_MM))[:, None], [70, 140, 255], [40, 220, 90]))
        px[v, u] = (0.35 * px[v, u] + 0.65 * col).astype(np.uint8)
        boxes.append((u.min(), v.min(), u.max(), v.max()))
    img = Image.fromarray(px)
    dr = ImageDraw.Draw(img)
    font = ImageFont.load_default(size=15)
    for (_, p), b in zip(picks.iterrows(), boxes):
        d = dets[int(p.det)]
        R, t = np.asarray(d["R"]), np.asarray(d["t"])
        o = K @ t; o = o[:2] / o[2]
        for ax, c in zip(range(3), [(255, 60, 60), (60, 255, 60), (70, 140, 255)]):
            e = K @ (t + R[:, ax] * 0.45 * model.diameter); e = e[:2] / e[2]
            dr.line([tuple(o), tuple(e)], fill=c, width=3)
        e = K @ np.array([p.cam_x_mm, p.cam_y_mm, p.cam_z_mm]); e = e[:2] / e[2]       # output position (top end)
        dr.ellipse([e[0] - 7, e[1] - 7, e[0] + 7, e[1] + 7], outline=(255, 230, 0), width=3)
        dr.text((b[2] - 10, b[1] - 4), f"#{p.name}", fill=(255, 230, 0), font=font, stroke_width=2, stroke_fill=(0, 0, 0))
    x0 = max(min(b[0] for b in boxes) - pad, 0); y0 = max(min(b[1] for b in boxes) - pad, 0)
    x1 = min(max(b[2] for b in boxes) + pad, w); y1 = min(max(b[3] for b in boxes) + pad, h)
    img = img.crop((x0, y0, x1, y1))
    ImageDraw.Draw(img).text((8, 6), f"{len(picks)} selected: CAD surface vs depth (green on depth / red see-through / "
                             "blue covered), yellow = output position (top end), axes x red y green z blue", fill=(255, 255, 255), font=font,
                             stroke_width=2, stroke_fill=(0, 0, 0))
    display(img.resize((width, int(width * img.height / img.width))))

draw_picks(f, dets, picks)

In [ ]:
cols_isaac = ["isaac_x_mm", "isaac_y_mm", "isaac_z_mm", "isaac_roll_deg", "isaac_pitch_deg", "isaac_yaw_deg"]
print("Isaac Sim world frame. Isaac Sim의 월드 원점 기준(0,0,0), x_m, y_m, z_m = TOP END POINT (yellow circle in the image), NOT the part centre: the point "
      "where the part axis leaves the end face that points up (hub end or flat face, column 'end'), 10.45 mm from the "
      "part centre along the axis. The part centre is in the CSV columns isaac_centre_x/y/z_mm.\n"
      "Orientation: roll / pitch / yaw in deg (fixed axes x -> y -> z, R = Rz(yaw) Ry(pitch) Rx(roll)). Part z = plate "
      "axis (+z towards the hub); the part is 2-fold symmetric about it, so a 180 deg turn about z is the same pose.")
isaac = picks[cols_isaac].copy()
isaac[["isaac_x_mm", "isaac_y_mm", "isaac_z_mm"]] /= 1000.0
isaac.columns = ["end_x_m", "end_y_m", "end_z_m", "roll_deg", "pitch_deg", "yaw_deg"]
isaac.insert(0, "end", picks["end"])
display(isaac.round(5))

print("Same top end points in the bin frame(앞쪽 6개는, 노란색 끝점을 부품이 담긴 상자(bin)를 기준으로 잰 좌표, 뒤쪽 6개는 카메라 기준으로 잰 좌표) (= scene_camera.json world) and the camera frame, mm:")
show = ["bin_x_mm", "bin_y_mm", "bin_z_mm", "bin_roll_deg", "bin_pitch_deg", "bin_yaw_deg",
        "cam_x_mm", "cam_y_mm", "cam_z_mm", "cam_roll_deg", "cam_pitch_deg", "cam_yaw_deg"]
display(picks[show].round(3))
print("Pose quality: label-free checks, and the error against the simulator GT (not available on a real robot):")
display(picks[["det", "end", "height_mm", "axis_tilt_deg", "covered", "depth_agree", "fitness", "rmse_mm",
               "gt_end_err_mm", "gt_centre_err_mm", "gt_tilt_err_deg", "gt_add_s_mm"]].round(3))

out_dir = Path(RUNS[DET]) / "pick_poses"
out_dir.mkdir(exist_ok=True)
picks.to_csv(out_dir / f"{SCENE}_{VIEW:06d}.csv", float_format="%.5f")
cand.to_csv(out_dir / f"{SCENE}_{VIEW:06d}_all_confident.csv", float_format="%.5f", index=False)
print(f"saved: {out_dir / f'{SCENE}_{VIEW:06d}.csv'} (top {len(picks)}), {out_dir / f'{SCENE}_{VIEW:06d}_all_confident.csv'} ({len(cand)})")

In [ ]:
# the same selection on every evaluated frame: are the exported poses right? (sim GT)
allp = []
run = load_run(RUNS[DET], cfg.refine)
gt_tab = tables[DET].set_index(["scene", "view", "det"])
for (s, v), ds in run.items():
    fr = load_frame(sim, sidx[(s, v)])
    Rw, tw = w2c(s, v)
    rows = []
    for n, d in enumerate(ds):
        if not d["confident"]:
            continue
        if covered_share(d["R"], d["t"], fr["depth"], fr["K"]) > MAX_COVERED:
            continue
        g = gt_tab.loc[(s, v, n)]
        rows.append({"scene": s, "view": v, "height_mm": (Rw.T @ (np.asarray(d["t"]) - tw))[2], "matched": g.matched,
                     "add_s": g.final_add_s if g.matched else np.nan, "centre": g.final_centre_mm if g.matched else np.nan,
                     "tilt": g.final_tilt_deg if g.matched else np.nan})
    allp += sorted(rows, key=lambda r: -r["height_mm"])[:TOP_N]
allp = pd.DataFrame(allp)
print(f"{len(allp)} exported poses over {allp.groupby(['scene', 'view']).ngroups} frames (top {TOP_N} per frame)")
display(pd.DataFrame({"not a real part": [int((~allp.matched).sum())],
                      "ADD-S < 1 mm": [f"{(allp.add_s < 1).mean():.1%}"],
                      "centre err median / max (mm)": [f"{allp.centre.median():.2f} / {allp.centre.max():.2f}"],
                      "axis tilt err median / max (deg)": [f"{allp.tilt.median():.2f} / {allp.tilt.max():.2f}"]},
                     index=["exported poses vs GT"]).T)

**결과 (§6):**
- scene 000000 view 0(부품 297개)에서 confident pose 75개 중 위가 비어 있는 23개가 남고, 그중 높은 8개를 출력했습니다.
  8개 모두 정답과 위치 0.07mm, 기울기 0.07° 이내입니다.
- 50장 전체에 같은 선택을 적용하면 400개를 출력합니다.
  - 정답과 매칭된 출력의 97.8%가 ADD-S 1mm 이내이고, **위치 오차는 최대 0.35mm, 판 축의 기울기 오차는 최대 1.1°**입니다.
  - 1mm를 넘는 2%도 평면 안 회전만 틀린 경우입니다.
  - 마스크 매칭이 안 된 9개는 마스크가 부분적인 실제 부품으로, 위치가 정답과 2.2mm 이내입니다.
- **선택된 부품은 많이 기울어 있습니다**(판 축이 수직에서 31~65°). 쌓인 bin의 맨 위 부품은 대개 다른 부품에 기대 있기 때문입니다.
  그리퍼의 접근 각도에 제한이 있다면 `axis_tilt_deg` 열로 거르거나 정렬 기준에 넣으세요.

## 7. 결론, 주의점

**결론: 시뮬레이션으로만 학습한 모델이 카메라 설정이 다른 시뮬레이션 데이터에서도 동작합니다.**
렌즈 왜곡을 보정한 뒤, 위가 비어 있는 부품의 6D pose를 위치 0.35mm, 축 기울기 1.1° 이내로 출력합니다.

| detector 마스크, 50장 (부품 8054개, 장당 약 160개) | 처음 | 깊이 맞춤 + 중복 제거 | + 깊이 검증 (현재) |
|---|---|---|---|
| 장당 confident pose | 56.3개 | 71.1개 | 71.1개 |
| confident 중 정답과 1mm 이내 | 99.2% | 98.1% | 98.1% |
| confident인데 틀림 (≥ 1mm) | 22개 | 67개 | 67개 (대부분 평면 안 회전) |
| 출력 중 오검출 | 224개 | 202개 | 31개 (30개는 마스크만 부분적인 실제 부품) |
| 가려지지 않은 부품(보이는 픽셀 > 5000) 중 confident이고 1mm 이내 | | | 96.1% |

**주의점**
- **렌즈 왜곡을 반드시 보정해야 합니다.** 이 데이터는 렌더링에 왜곡이 들어 있어 `undistort_bop.py`로 보정한 사본을 썼습니다.
  Isaac Sim에서 직접 찍을 때도 카메라에 왜곡이 있다면 같은 보정이 필요합니다(`camera.json`의 `dist`).
- **평면 안 회전 오류(confident인데 틀림의 대부분)는 깊이 검증으로 걸러지지 않습니다.** 위치와 판 방향은 맞으므로, 중심을 집는 그리퍼라면 영향이 작습니다.
  회전 방향이 중요한 작업이라면 별도 확인이 필요합니다.
- **Isaac Sim 좌표는 `config_resolved.yaml`의 bin 위치를 기준으로 계산했습니다.** 실제 Isaac Sim 장면에서 bin 위치가 다르면 §6의 `bin_pos_mm`, `bin_rot`를 바꾸세요.
- **시뮬레이션 depth는 실제보다 깨끗합니다.** 실제 센서에서는 정확도와 confident 수가 낮아집니다(`results_photoneo_summary.ipynb` 참고).